In [1]:
import ipywidgets as wdg
from ipywidgets import interact

import pandas as pd
import numpy as np
import folium
import branca.colormap as cm
import matplotlib.pyplot as plt
import datetime

In [2]:
history_toulouse_test = pd.read_parquet("data/history_toulouse_test.parquet")
history_toulouse_train = pd.read_parquet("data/history_toulouse_train.parquet")
stations_toulouse = pd.read_parquet("data/stations_toulouse.parquet")

In [3]:
stations_toulouse.pop("station_id")

0        0
1        1
2        2
3        3
4        4
      ... 
858    858
859    859
860    860
861    861
862    862
Name: station_id, Length: 863, dtype: int64

In [4]:
# centre de la carte

lat_centre = stations_toulouse["latitude"].median()
lon_centre = stations_toulouse["longitude"].median()
print(lat_centre, lon_centre)
# capitole de toulouse
"""lat_centre = 43.604429
lon_centre = 1.443812"""


# formule de calcul des distance
def distance_by_center(lat_c, lon_c, lat, lon):
    # conversion en randians
    lat_c, lon_c, lat, lon = map(np.radians, [lat_c, lon_c, lat, lon])

    # formule proximité plane(pythagore)

    dlat = lat_c - lat
    dlon = (lon_c - lon) * np.cos((lat_c + lat) / 2)

    distance = 6371 * np.sqrt(dlat**2 + dlon**2)

    return distance

43.601607 1.44293


In [5]:
# creation des groupes
stations_toulouse["distance_centre"] = distance_by_center(
    lat_centre,
    lon_centre,
    stations_toulouse["latitude"],
    stations_toulouse["longitude"],
)

stations_toulouse["groupe"] = pd.qcut(
    stations_toulouse["distance_centre"], q=3, labels=["centre", "milieu", "exterieur"]
)


stations_toulouse.pop("distance_centre")

print(stations_toulouse)

                               station  longitude   latitude     groupe
0             00001 - POIDS DE L'HUILE   1.445475  43.604190     centre
1                    00002 - LAFAYETTE   1.445306  43.604885     centre
2    00003 - ALSACE - LORRAINE - POMME   1.445454  43.603302     centre
3      00003 - ALSACE-LORRAINE - POMME   1.445454  43.603302     centre
4                        00003 - POMME   1.445454  43.603302     centre
..                                 ...        ...        ...        ...
858            TOLOSANE - PASTOURELLES   1.477417  43.543115  exterieur
859               TOLOSANE - POUCIQUOT   1.469875  43.551101  exterieur
860               TOURNEFEUILLE MAIRIE   1.346042  43.583665  exterieur
861                VELASQUEZ - TAMARIS   1.390347  43.621004  exterieur
862                        ZOLA - DALÍ   1.371961  43.644069  exterieur

[863 rows x 4 columns]


In [6]:
# ajout du taux de remplissage

history_toulouse_train["taux_remplissage"] = (
    history_toulouse_train["bikes"]
    / (history_toulouse_train["bikes"] + history_toulouse_train["stands"])
    * 100
)
history_toulouse_train["taux_remplissage"] = history_toulouse_train[
    "taux_remplissage"
].round(1)

print(history_toulouse_train)

                                              bikes  stands  taux_remplissage
station                  commit_at                                           
00001 - POIDS DE L'HUILE 2023-08-05 12:15:00     10       9              52.6
                         2023-08-05 12:30:00     10       9              52.6
                         2023-08-05 12:45:00     10       9              52.6
                         2023-08-05 13:00:00     11       8              57.9
                         2023-08-05 13:15:00     11       8              57.9
...                                             ...     ...               ...
TEST BEE                 2026-02-12 15:00:00      0       3               0.0
                         2026-02-12 15:15:00      0       3               0.0
                         2026-02-12 15:30:00      0       3               0.0
                         2026-02-12 15:45:00      0       3               0.0
                         2026-02-12 16:00:00      0       3     

In [7]:
"""fonction qui calcule les deltacos/deltasin a partir du dernier temps enregistrer t et t+h, h etant l'horizon de temps(15,30,45,60,etc...)
periode de la courbe = 96 (je prend le quart d'heure en unité donc 24*15=96)
ordre de la fonction de base de fourier
t1 : dernier temps enregistrer en quart d'heure
t2 : t1+h


"""


def delta_Fourier(ordre, t1, t2):
    delta_cos = np.cos(ordre * 2 * np.pi * t2 / 96) - np.cos(
        ordre * 2 * np.pi * t1 / 96
    )

    delta_sin = np.sin(ordre * 2 * np.pi * t2 / 96) - np.sin(
        ordre * 2 * np.pi * t1 / 96
    )

    return delta_cos, delta_sin

In [8]:
from sklearn.linear_model import LinearRegression

modele = LinearRegression()

modele

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [9]:
derniere_horaire = history_toulouse_train.index.get_level_values(level="commit_at")

df_base = pd.DataFrame(
    {
        "derniere horaire(decimal 15min)": (
            derniere_horaire - derniere_horaire.normalize()
        )
        / pd.Timedelta(minutes=15),
        "taux_remplissage": history_toulouse_train["taux_remplissage"],
    },
    index=history_toulouse_train.index,
)


print(df_base)

                                              derniere horaire(decimal 15min)  \
station                  commit_at                                              
00001 - POIDS DE L'HUILE 2023-08-05 12:15:00                             49.0   
                         2023-08-05 12:30:00                             50.0   
                         2023-08-05 12:45:00                             51.0   
                         2023-08-05 13:00:00                             52.0   
                         2023-08-05 13:15:00                             53.0   
...                                                                       ...   
TEST BEE                 2026-02-12 15:00:00                             60.0   
                         2026-02-12 15:15:00                             61.0   
                         2026-02-12 15:30:00                             62.0   
                         2026-02-12 15:45:00                             63.0   
                         202

In [10]:
# creation de la matrice feature/target
models = {}  # dictionaire de modele (groupe, horyzon)
for groupe in stations_toulouse["groupe"].unique():

    print(groupe)
    stations_groupe = stations_toulouse.loc[
        stations_toulouse["groupe"] == groupe, "station"
    ].values

    df_groupe = df_base.loc[
        df_base.index.get_level_values("station").isin(stations_groupe)
    ]
    for h in range(1, 9):
        df_h = df_groupe.copy()
        for i in range(1, 4):
            df_h[f"Dcos_{i}"], df_h[f"Dsin_{i}"] = delta_Fourier(
                i,
                df_h["derniere horaire(decimal 15min)"],
                df_h["derniere horaire(decimal 15min)"] + h,
            )

        # les index des reel donnée après l'horyzon h
        index_target = pd.MultiIndex.from_arrays(
            [
                df_h.index.get_level_values("station"),
                df_h.index.get_level_values("commit_at") + pd.Timedelta(minutes=15 * h),
            ]
        )

        df_h["target"] = (
            history_toulouse_train["taux_remplissage"].reindex(index_target).values
        )
        df_h = df_h.dropna(subset=["target", "taux_remplissage"])

        # feature et target
        X = df_h.drop(columns=["derniere horaire(decimal 15min)", "target"])
        Y = df_h["target"]
        models[(groupe, h)] = LinearRegression().fit(X, Y)

centre
milieu
exterieur


**creation de history_toulouse_apprentissage**

In [11]:
# Exemple de création d'un jeu de données d'apprentissage et un jeu de données de validation

# Création ensemble d'apprentissage
# Suppression des 8 dernières mesures de chaque station
rang_depuis_la_fin = history_toulouse_train.groupby(level="station").cumcount(
    ascending=False
)
rang_depuis_la_fin

station                   commit_at          
00001 - POIDS DE L'HUILE  2023-08-05 12:15:00    104774
                          2023-08-05 12:30:00    104773
                          2023-08-05 12:45:00    104772
                          2023-08-05 13:00:00    104771
                          2023-08-05 13:15:00    104770
                                                  ...  
TEST BEE                  2026-02-12 15:00:00         4
                          2026-02-12 15:15:00         3
                          2026-02-12 15:30:00         2
                          2026-02-12 15:45:00         1
                          2026-02-12 16:00:00         0
Length: 40012622, dtype: int64

In [12]:
history_toulouse_apprentissage = history_toulouse_train[rang_depuis_la_fin >= 8].copy()
history_toulouse_apprentissage

bikes  stands  taux_remplissage
station                  commit_at                                           
00001 - POIDS DE L'HUILE 2023-08-05 12:15:00     10       9              52.6
                         2023-08-05 12:30:00     10       9              52.6
                         2023-08-05 12:45:00     10       9              52.6
                         2023-08-05 13:00:00     11       8              57.9
                         2023-08-05 13:15:00     11       8              57.9
...                                             ...     ...               ...
TEST BEE                 2026-02-12 13:00:00      0       3               0.0
                         2026-02-12 13:15:00      0       3               0.0
                         2026-02-12 13:30:00      0       3               0.0
                         2026-02-12 13:45:00      0       3               0.0
                         2026-02-12 14:00:00      0       3               0.0

[40006230 rows x 3 columns]

In [13]:
# Récupèration de la dernière date de mesure pour chaque station
# des données d'apprentissage
last_apprentissage = (
    history_toulouse_apprentissage.groupby(level="station").tail(1).copy()
)

last_apprentissage

,,bikes,stands,taux_remplissage
station,commit_at,,,
00001 - POIDS DE L'HUILE,2026-07-31 19:45:00,6,10,37.5
00002 - LAFAYETTE,2026-07-31 21:45:00,5,13,27.8
00003 - ALSACE - LORRAINE - POMME,2025-11-10 09:45:00,7,11,38.9
00003 - ALSACE-LORRAINE - POMME,2026-07-31 19:45:00,15,1,93.8
00003 - POMME,2025-01-05 03:45:00,17,0,100.0
...,...,...,...,...
MONT LOUIS –AV. DE TOULOUSE,2026-03-23 18:15:00,10,5,66.7
NARBONNE - CLOTASSES,2025-08-22 17:15:00,0,16,0.0
PASTEUR - LAVOISIER,2025-08-05 08:15:00,0,0,NaN


In [14]:
# centre de la carte

lat_centre = stations_toulouse["latitude"].median()
lon_centre = stations_toulouse["longitude"].median()
print(lat_centre, lon_centre)
# capitole de toulouse


# formule de calcul des distance
def distance_by_center(lat_c, lon_c, lat, lon):
    # conversion en randians
    lat_c, lon_c, lat, lon = map(np.radians, [lat_c, lon_c, lat, lon])

    # formule proximité plane(pythagore)

    dlat = lat_c - lat
    dlon = (lon_c - lon) * np.cos((lat_c + lat) / 2)

    distance = 6371 * np.sqrt(dlat**2 + dlon**2)

    return distance

43.601607 1.44293


In [15]:
index_heure = history_toulouse_apprentissage.index.get_level_values("commit_at")


heure = index_heure.hour.astype(str).str.zfill(2)
minute = index_heure.minute.astype(str).str.zfill(2)

history_toulouse_apprentissage["horaire"] = heure + ":" + minute

In [16]:
# creation des groupes
stations_toulouse["distance_centre"] = distance_by_center(
    lat_centre,
    lon_centre,
    stations_toulouse["latitude"],
    stations_toulouse["longitude"],
)

stations_toulouse["groupe"] = pd.qcut(
    stations_toulouse["distance_centre"], q=3, labels=["centre", "milieu", "exterieur"]
)
print(stations_toulouse)

                               station  longitude   latitude     groupe  \
0             00001 - POIDS DE L'HUILE   1.445475  43.604190     centre   
1                    00002 - LAFAYETTE   1.445306  43.604885     centre   
2    00003 - ALSACE - LORRAINE - POMME   1.445454  43.603302     centre   
3      00003 - ALSACE-LORRAINE - POMME   1.445454  43.603302     centre   
4                        00003 - POMME   1.445454  43.603302     centre   
..                                 ...        ...        ...        ...   
858            TOLOSANE - PASTOURELLES   1.477417  43.543115  exterieur   
859               TOLOSANE - POUCIQUOT   1.469875  43.551101  exterieur   
860               TOURNEFEUILLE MAIRIE   1.346042  43.583665  exterieur   
861                VELASQUEZ - TAMARIS   1.390347  43.621004  exterieur   
862                        ZOLA - DALÍ   1.371961  43.644069  exterieur   

     distance_centre  
0           0.352828  
1           0.411655  
2           0.277178  
3      

In [34]:
history_toulouse_apprentissage["groupe"] = (
    history_toulouse_apprentissage.index.get_level_values("station").map(
        stations_toulouse.set_index("station")["groupe"]
    )
)
print(history_toulouse_apprentissage)
print(
    history_toulouse_apprentissage.loc[
        history_toulouse_apprentissage[["groupe", "horaire"]].isna(),
        ["horaire", "groupe"],
    ]
)

                                              bikes  stands  taux_remplissage  \
station                  commit_at                                              
00001 - POIDS DE L'HUILE 2023-08-05 12:15:00     10       9              52.6   
                         2023-08-05 12:30:00     10       9              52.6   
                         2023-08-05 12:45:00     10       9              52.6   
                         2023-08-05 13:00:00     11       8              57.9   
                         2023-08-05 13:15:00     11       8              57.9   
...                                             ...     ...               ...   
TEST BEE                 2026-02-12 13:00:00      0       3               0.0   
                         2026-02-12 13:15:00      0       3               0.0   
                         2026-02-12 13:30:00      0       3               0.0   
                         2026-02-12 13:45:00      0       3               0.0   
                         202

ValueError: Cannot index with multidimensional key

In [35]:
stat = (
    history_toulouse_apprentissage.groupby(["horaire", "groupe"])[
        "taux_remplissage"
    ].agg(
        moyenne="mean",
        mediane="median",
        q1=lambda x: x.quantile(0.25),
        q3=lambda x: x.quantile(0.75),
    )
    # .quantile([0.25,0.5,0.75],"higher")
    # .unstack()
)
stat

moyenne  mediane    q1    q3
horaire groupe                                   
00:00   centre     30.566737     21.1   4.5  51.6
        milieu     48.417248     45.5  22.2  73.7
        exterieur  42.610219     38.5  16.7  66.7
00:15   centre     30.268966     20.0   4.3  51.5
        milieu     48.967973     46.2  23.1  75.0
...                      ...      ...   ...   ...
23:30   milieu     47.072786     44.4  21.1  71.4
        exterieur  41.778439     37.5  15.4  65.0
23:45   centre     30.946776     22.2   4.8  52.2
        milieu     47.769170     45.0  22.2  72.7
        exterieur  42.189127     37.5  15.8  66.7

[288 rows x 4 columns]

In [36]:
import utils.previsions_method as method

In [49]:
def station_a_predire(df_a_predire):
    # copie de securité du datframe des predictions à faire
    df_predire_cpy = df_a_predire.copy()
    horaire_test = df_predire_cpy.index.get_level_values("commit_at")

    # creation de la capacité max
    total_capacity = (
        (
            history_toulouse_apprentissage["bikes"]
            + history_toulouse_apprentissage["stands"]
        )
        .reindex(df_a_predire.index)
        .values
    )
    # creation du dataframe servant de base : la derniere horaire disponible, le taux de remplissage et la capcité total de chaque stations
    df_base_feature = pd.DataFrame(
        {
            "derniere horaire(decimal 15min)": (horaire_test - horaire_test.normalize())
            / pd.Timedelta(minutes=15),
            "taux_remplissage": history_toulouse_apprentissage["taux_remplissage"]
            .reindex(df_a_predire.index)
            .values,
            "capacité": total_capacity,
        },
        index=df_a_predire.index,
    )

    profil_moyen = method.mean_by_profil(df_a_predire, history_toulouse_apprentissage)
    # creation des previsions pour chaque groupe

    # filtre pour les stations avec trop peu de donnée
    empty_station = df_base_feature.notna().all(axis=1).to_numpy()

    for groupe in stations_toulouse["groupe"].unique():

        # masque pour selectionner un groupe
        station_group = stations_toulouse.loc[
            stations_toulouse["groupe"] == groupe, "station"
        ].values

        mask = (
            df_base_feature.index.get_level_values("station").isin(station_group)
            & empty_station
        )

        # dataframe comportant seulement le groupe a remplir sur ce tour de boucle utiliser pour la regression lineaire
        df_current_group = df_base_feature.loc[mask]

        # pour chaque horizon de temps, on predit avec le bon algorithme
        # ajout des lignes des fonctions de base fourier (cos et sin)(serie de fourier d'ordre 3)
        for h in range(1, 9):
            df_h_test = df_current_group.copy()
            # calcul des delta cos et sin des serie de fourier
            for i in range(1, 4):
                df_h_test[f"Dcos_{i}"], df_h_test[f"Dsin_{i}"] = delta_Fourier(
                    i,
                    df_h_test["derniere horaire(decimal 15min)"],
                    df_h_test["derniere horaire(decimal 15min)"] + h,
                )

            X_to_predicts = df_h_test.drop(
                columns=["derniere horaire(decimal 15min)", "capacité"]
            )

            # prediciton en taux (pourcentage)
            prediction_taux = np.clip(
                models[(groupe, h)].predict(X_to_predicts), 0, 100
            )

            # prediction en nombre de velo
            prediction_velo = (prediction_taux / 100) * df_h_test["capacité"].values

            # ajout des predictions realisé dans le dataframe donné
            df_predire_cpy.loc[mask, f"bikes_+{h*15}min"] = prediction_velo

    # remplissage des lignes non restante non remplie par une methode plus simple
    df_predire_cpy = df_predire_cpy.fillna(profil_moyen)
    df_predire_cpy = df_predire_cpy.fillna(7)

    return df_predire_cpy

**calcul RMSE**

In [38]:
# Creation de l'ensemble de validation
# On récupère les 9 dernières mesures :
# - la 1ère = dernière mesure dans l'ensemble d'apprentissage
# - les 8 suivantes = valeurs à prédire
last_9 = history_toulouse_train.groupby(level="station").tail(9).copy()
last_9

bikes  stands  taux_remplissage
station                  commit_at                                           
00001 - POIDS DE L'HUILE 2026-07-31 19:45:00      6      10              37.5
                         2026-07-31 20:00:00      1      15               6.2
                         2026-07-31 20:15:00      1      15               6.2
                         2026-07-31 20:30:00      1      15               6.2
                         2026-07-31 20:45:00      1      15               6.2
...                                             ...     ...               ...
TEST BEE                 2026-02-12 15:00:00      0       3               0.0
                         2026-02-12 15:15:00      0       3               0.0
                         2026-02-12 15:30:00      0       3               0.0
                         2026-02-12 15:45:00      0       3               0.0
                         2026-02-12 16:00:00      0       3               0.0

[7189 rows x 3 columns]

In [39]:
# On remet l'index sous forme de colonnes
last_9 = last_9.reset_index()
last_9

,station,commit_at,bikes,stands,taux_remplissage
0,00001 - POIDS DE L'HUILE,2026-07-31 19:45:00,6,10,37.5
1,00001 - POIDS DE L'HUILE,2026-07-31 20:00:00,1,15,6.2
2,00001 - POIDS DE L'HUILE,2026-07-31 20:15:00,1,15,6.2
3,00001 - POIDS DE L'HUILE,2026-07-31 20:30:00,1,15,6.2
4,00001 - POIDS DE L'HUILE,2026-07-31 20:45:00,1,15,6.2
...,...,...,...,...,...
7184,TEST BEE,2026-02-12 15:00:00,0,3,0.0
7185,TEST BEE,2026-02-12 15:15:00,0,3,0.0
7186,TEST BEE,2026-02-12 15:30:00,0,3,0.0
7187,TEST BEE,2026-02-12 15:45:00,0,3,0.0


In [40]:
last_9["horizon"] = last_9.groupby("station").cumcount()

In [41]:
# Recherche de valeurs manquantes
missing = last_9.loc[
    last_9[["bikes", "stands"]].isna().any(axis=1),
    ["station", "commit_at", "bikes", "stands"],
]

print(missing)

Empty DataFrame
Columns: [station, commit_at, bikes, stands]
Index: []


In [42]:
# On crée une ligne par station et on récupère les 8 valeurs futures
history_toulouse_validation = (
    last_9.pivot(index="station", columns="horizon", values="bikes")
    .drop(columns=0)
    .rename(
        columns={
            1: "bikes_+15min",
            2: "bikes_+30min",
            3: "bikes_+45min",
            4: "bikes_+60min",
            5: "bikes_+75min",
            6: "bikes_+90min",
            7: "bikes_+105min",
            8: "bikes_+120min",
        }
    )
)

# Récupérer la date de la dernière mesure de l'ensemble d'apprentissage par station
cutoff_times = last_9[last_9["horizon"] == 0].set_index("station")["commit_at"]

# Ajouter commit_at
history_toulouse_validation["commit_at"] = cutoff_times

# Mettre station + commit_at en MultiIndex
history_toulouse_validation = (
    history_toulouse_validation.reset_index()
    .set_index(["station", "commit_at"])
    .sort_index()
)

history_toulouse_validation

,horizon,bikes_+15min,bikes_+30min,bikes_+45min,bikes_+60min,bikes_+75min,bikes_+90min,bikes_+105min,bikes_+120min
station,commit_at,,,,,,,,
00001 - POIDS DE L'HUILE,2026-07-31 19:45:00,1.0,1.0,1.0,1.0,1.0,1.0,9.0,9.0
00002 - LAFAYETTE,2026-07-31 21:45:00,5.0,5.0,5.0,9.0,9.0,9.0,9.0,9.0
00003 - ALSACE - LORRAINE - POMME,2025-11-10 09:45:00,9.0,12.0,15.0,16.0,17.0,17.0,17.0,18.0
00003 - ALSACE-LORRAINE - POMME,2026-07-31 19:45:00,0.0,0.0,0.0,0.0,0.0,0.0,12.0,12.0
00003 - POMME,2025-01-05 03:45:00,17.0,17.0,17.0,17.0,17.0,17.0,17.0,17.0
...,...,...,...,...,...,...,...,...,...
NARBONNE - CLOTASSES,2025-08-22 17:15:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
PASTEUR - LAVOISIER,2025-08-05 08:15:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
PLACE GEORGES BRASSENS,2025-08-05 09:15:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN


**creation des donnée a predire**

In [43]:
# Creation des données à prédire
history_toulouse_basique = history_toulouse_validation.copy()

In [ ]:
history_toulouse_basique = station_a_predire(history_toulouse_basique)
print(history_toulouse_basique.dtypes)
print(history_toulouse_basique.isna().sum())

horizon                                                bikes_+15min  \
station                           commit_at                           
00001 - POIDS DE L'HUILE          2026-07-31 19:45:00      5.990830   
00002 - LAFAYETTE                 2026-07-31 21:45:00      5.018533   
00003 - ALSACE - LORRAINE - POMME 2025-11-10 09:45:00      7.059437   
00003 - ALSACE-LORRAINE - POMME   2026-07-31 19:45:00     14.705986   
00003 - POMME                     2025-01-05 03:45:00     16.569817   
...                                                             ...   
NARBONNE - CLOTASSES              2025-08-22 17:15:00      0.063228   
PASTEUR - LAVOISIER               2025-08-05 08:15:00      0.000000   
PLACE GEORGES BRASSENS            2025-08-05 09:15:00      0.000000   
SERVANTY AIRBUS TRAMWAY           2025-08-01 10:15:00      0.129264   
TEST BEE                          2026-02-12 14:00:00      0.015017   

horizon                                                bikes_+30min  \
stati

In [53]:
# Calcul de l'erreur RMSE sur l'ensemble de validation
rmse = np.sqrt(
    np.nanmean(
        (
            history_toulouse_basique.values
            - history_toulouse_validation.loc[history_toulouse_basique.index].values
        )
        ** 2
    )
)

print(f"RMSE : {rmse:.6f}")

RMSE : 2.172378


In [ ]:
history_toulouse_basique.to_parquet(
    path="prevision/prediction_geo.parquet", engine="auto", index=False
)